In [1]:
import numpy as np
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import shap
from xgboost import XGBClassifier
# import random forest regressor
from sklearn.ensemble import RandomForestRegressor
#import linear regression
from sklearn.linear_model import LinearRegression
# import tqdm
from tqdm import tqdm
import tqdm
#import r2_score
from sklearn.metrics import r2_score
#import confusion matrix
from sklearn.metrics import confusion_matrix
# import roc auc score
from sklearn.metrics import roc_auc_score
from food_crisis_functions import *
import json
with open("forecasting_hyperparameters.json", "r") as file:
    best_params_xgb_regressor= json.load(file)
    
with open("forecasting_hyperparameters_p3.json", "r") as file:
    best_params_xgb_regressor_for_p3= json.load(file)

In [2]:

# read csv
df = pd.read_csv(r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\IPCCH_2017_2025_final_v12102025_with_zscores.csv')
# add dummys for area_id and month year
#df = pd.concat([df, pd.get_dummies(df['area_id'], prefix='area_id')], axis=1)
#df = pd.concat([df, pd.get_dummies(df['date'], prefix='month_year')], axis=1)
# drop lat and lon
#df = df.drop(['lat', 'lon'], axis=1)
###drop fews_ipc_ha
#df = df.drop(['fews_ipc_ha'], axis=1)
# random split train and test


In [3]:
# drop all variable with suffix _m12, _v12, _l12
df = df.drop(df.filter(regex='(_m12|_v12|_l12)$').columns, axis=1)

In [4]:
df.columns.tolist()

['AEZ_32000',
 'AEZ_34000',
 'AEZ_36000',
 'AEZ_38000',
 'AEZ_42000',
 'AEZ_7000',
 'CC',
 'CPI',
 'EVI_mean',
 'EVI_std',
 'Evap_tavg_mean',
 'Evap_tavg_stdDev',
 'FAO_price',
 'Food_CPI',
 'Food_food_inflation',
 'GDP',
 'GPP_mean',
 'GPP_std',
 'ISO3',
 'LWdown_f_tavg_mean',
 'LWdown_f_tavg_stdDev',
 'Lwnet_tavg_mean',
 'Lwnet_tavg_stdDev',
 'Psurf_f_tavg_mean',
 'Psurf_f_tavg_stdDev',
 'Qair_f_tavg_mean',
 'Qair_f_tavg_stdDev',
 'Qg_tavg_mean',
 'Qg_tavg_stdDev',
 'Qh_tavg_mean',
 'Qh_tavg_stdDev',
 'Qle_tavg_mean',
 'Qle_tavg_stdDev',
 'Qs_tavg_mean',
 'Qs_tavg_stdDev',
 'Qsb_tavg_mean',
 'Qsb_tavg_stdDev',
 'RadT_tavg_mean',
 'RadT_tavg_stdDev',
 'Rainf_f_tavg_mean',
 'Rainf_f_tavg_stdDev',
 'Rainf_zscore',
 'SWE_inst_mean',
 'SWE_inst_stdDev',
 'SWdown_f_tavg_mean',
 'SWdown_f_tavg_stdDev',
 'SnowCover_inst_mean',
 'SnowCover_inst_stdDev',
 'SnowDepth_inst_mean',
 'SnowDepth_inst_stdDev',
 'Snowf_tavg_mean',
 'Snowf_tavg_stdDev',
 'SoilMoi00_10cm_tavg_mean',
 'SoilMoi00_10cm_tav

In [5]:
from itertools import product

In [6]:
# extract unique admin_code, then create a dataframe of year 2022, 2023, 2024, with each month
unique_admin_codes = df['admin_code'].unique()
year = [2020, 2021, 2022, 2023, 2024,2025]
month = [1,2,3,4,5,6,7,8,9,10,11,12]


# Create combinations of admin_code, year, and month
combinations = list(product(unique_admin_codes, year, month))

# Create the dataframe
df_forecast = pd.DataFrame(combinations, columns=['admin_code', 'year', 'month'])

# Add a date column for convenience
df_forecast['date'] = pd.to_datetime(
    df_forecast[['year', 'month']].assign(day=1)
)


In [7]:
df['date'] = pd.to_datetime(
    df[['year', 'month']].assign(day=1)
)

In [8]:
# merge on date and admin_code
df_forecast = df_forecast.merge(df, on=['date', 'admin_code','year','month'], how='left')

In [9]:
time_invariant = ['AEZ_32000',
 'AEZ_34000',
 'AEZ_36000',
 'AEZ_38000',
 'AEZ_42000',
 'AEZ_7000',
 'crop',
 'distance_to_river',
 'elevation',
 'estimated_population',
 'lat',
 'lon',
 'market_access',
 'popdensity',
 'range',
 'ruggedness',
 'sg_cec_5-15cm',
 'sg_cfvo_5-15cm',
 'sg_nitrogen_5-15cm',
 'sg_phh2o_5-15cm',
 'sg_soc_5-15cm',
 'slope',
 'AEZ_10000',
 'AEZ_12000',
 'AEZ_17000',
 'AEZ_19000',
 'AEZ_20000',
 'AEZ_25000',
 'AEZ_28000',
 'AEZ_30000',
 'AEZ_31000',
 'AEZ_33000',
 'AEZ_35000',
 'AEZ_4000',
 'AEZ_40000',
 'AEZ_43000',
 'AEZ_9000']

time_variant = [  'nightlight_mean',
 'nightlight_std','market_distance', 'gini', 'event_count_battles',
 'event_count_battles_w10',
 'event_count_battles_w5',
 'event_count_explosions',
 'event_count_explosions_w10',
 'event_count_explosions_w5',
 'event_count_violence',
 'event_count_violence_w10',
 'event_count_violence_w5','CC',
 'CPI',
 'EVI_mean',
 'EVI_std',
 'Evap_tavg_mean',
 'Evap_tavg_stdDev',
 'FAO_price',
 'Food_CPI',
 'Food_food_inflation',
 'GDP',
 'GPP_mean',
 'GPP_std',
 'LWdown_f_tavg_mean',
 'LWdown_f_tavg_stdDev',
 'Lwnet_tavg_mean',
 'Lwnet_tavg_stdDev',
 'Psurf_f_tavg_mean',
 'Psurf_f_tavg_stdDev',
 'Qair_f_tavg_mean',
 'Qair_f_tavg_stdDev',
 'Qg_tavg_mean',
 'Qg_tavg_stdDev',
 'Qh_tavg_mean',
 'Qh_tavg_stdDev',
 'Qle_tavg_mean',
 'Qle_tavg_stdDev',
 'Qs_tavg_mean',
 'Qs_tavg_stdDev',
 'Qsb_tavg_mean',
 'Qsb_tavg_stdDev',
 'RadT_tavg_mean',
 'RadT_tavg_stdDev',
 'Rainf_f_tavg_mean',
 'Rainf_f_tavg_stdDev',
 'Rainf_zscore',
 'SWE_inst_mean',
 'SWE_inst_stdDev',
 'SWdown_f_tavg_mean',
 'SWdown_f_tavg_stdDev',
 'SnowCover_inst_mean',
 'SnowCover_inst_stdDev',
 'SnowDepth_inst_mean',
 'SnowDepth_inst_stdDev',
 'Snowf_tavg_mean',
 'Snowf_tavg_stdDev',
 'SoilMoi00_10cm_tavg_mean',
 'SoilMoi00_10cm_tavg_stdDev',
 'SoilMoi100_200cm_tavg_mean',
 'SoilMoi100_200cm_tavg_stdDev',
 'SoilMoi10_40cm_tavg_mean',
 'SoilMoi10_40cm_tavg_stdDev',
 'SoilMoi40_100cm_tavg_mean',
 'SoilMoi40_100cm_tavg_stdDev',
 'SoilTemp00_10cm_tavg_mean',
 'SoilTemp00_10cm_tavg_stdDev',
 'SoilTemp100_200cm_tavg_mean',
 'SoilTemp100_200cm_tavg_stdDev',
 'SoilTemp10_40cm_tavg_mean',
 'SoilTemp10_40cm_tavg_stdDev',
 'SoilTemp40_100cm_tavg_mean',
 'SoilTemp40_100cm_tavg_stdDev',
 'Swnet_tavg_mean',
 'Swnet_tavg_stdDev',
 'Tair_f_tavg_mean',
 'Tair_f_tavg_stdDev',
 'Tair_zscore',
 'WFP_Price',
 'WFP_Price_std',
 'Wind_f_tavg_mean',
 'Wind_f_tavg_stdDev',  'distance_to_nearest_acled',   'sum_fatalities_battles',
 'sum_fatalities_battles_w10',
 'sum_fatalities_battles_w5',
 'sum_fatalities_explosions',
 'sum_fatalities_explosions_w10',
 'sum_fatalities_explosions_w5',
 'sum_fatalities_violence',
 'sum_fatalities_violence_w10',
 'sum_fatalities_violence_w5', 
]

drop = [ 
 'ISO3', 'address', 
 'country',
 'country_code',
 'country_en', 'state']

In [10]:
#df_forecast drop columns drop
df_forecast = df_forecast.drop(drop, axis=1)

In [11]:
def impute_time_invariant_by_admin(df, time_invariant_cols, admin_col='admin_code', method='mean'):
    """
    Impute missing values in time-invariant features using admin_code level statistics.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe
    time_invariant_cols : list
        List of time-invariant column names to impute
    admin_col : str
        Admin code column name (default: 'admin_code')
    method : str
        Aggregation method: 'mean', 'median', 'mode', 'forward_fill'
    
    Returns:
    --------
    pd.DataFrame
        Dataframe with imputed values
    """
    df_imputed = df.copy()
    
    for col in time_invariant_cols:
        print(f"\nImputing {col}:")
        missing_before = df_imputed[col].isna().sum()
        print(f"  Missing values before: {missing_before} ({missing_before/len(df_imputed)*100:.2f}%)")
        
        if method == 'mean':
            # Calculate admin-level mean
            admin_stats = df_imputed.groupby(admin_col)[col].mean()
        elif method == 'median':
            # Calculate admin-level median
            admin_stats = df_imputed.groupby(admin_col)[col].median()
        elif method == 'mode':
            # Calculate admin-level mode
            admin_stats = df_imputed.groupby(admin_col)[col].agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else np.nan)
        elif method == 'forward_fill':
            # Forward fill within each admin_code group
            df_imputed[col] = df_imputed.groupby(admin_col)[col].ffill()
            missing_after = df_imputed[col].isna().sum()
            print(f"  Missing values after: {missing_after} ({missing_after/len(df_imputed)*100:.2f}%)")
            continue
        
        # Map the statistics back to fill missing values
        df_imputed[col] = df_imputed.apply(
            lambda row: admin_stats[row[admin_col]] if pd.isna(row[col]) and row[admin_col] in admin_stats.index 
            else row[col], 
            axis=1
        )
        
        missing_after = df_imputed[col].isna().sum()
        print(f"  Missing values after: {missing_after} ({missing_after/len(df_imputed)*100:.2f}%)")
        print(f"  Imputed: {missing_before - missing_after} values")
    
    return df_imputed

In [12]:
df_forecast = impute_time_invariant_by_admin(
    df_forecast, 
    time_invariant_cols=time_invariant,
    admin_col='admin_code',
    method='mode'  # or 'median', 'mode', 'forward_fill'
)


Imputing AEZ_32000:
  Missing values before: 74724 (16.67%)
  Missing values after: 0 (0.00%)
  Imputed: 74724 values

Imputing AEZ_34000:
  Missing values before: 74724 (16.67%)
  Missing values after: 0 (0.00%)
  Imputed: 74724 values

Imputing AEZ_36000:
  Missing values before: 74724 (16.67%)
  Missing values after: 0 (0.00%)
  Imputed: 74724 values

Imputing AEZ_38000:
  Missing values before: 74724 (16.67%)
  Missing values after: 0 (0.00%)
  Imputed: 74724 values

Imputing AEZ_42000:
  Missing values before: 74724 (16.67%)
  Missing values after: 0 (0.00%)
  Imputed: 74724 values

Imputing AEZ_7000:
  Missing values before: 74724 (16.67%)
  Missing values after: 0 (0.00%)
  Imputed: 74724 values

Imputing crop:
  Missing values before: 74724 (16.67%)
  Missing values after: 0 (0.00%)
  Imputed: 74724 values

Imputing distance_to_river:
  Missing values before: 74724 (16.67%)
  Missing values after: 0 (0.00%)
  Imputed: 74724 values

Imputing elevation:
  Missing values before: 

In [13]:

# create _l12 and _v12 for each time_variant
# _l12 = value from 12 months ago
# _v12 = 12-month moving average lagged 12 months (i.e., average of months 13-24 ago)

import pandas as pd

def create_lag_features(df, time_variant_cols, admin_col='admin_code', 
                        time_col=None, year_col='year', month_col='month'):
    """
    Create _l12 and _v12 features for each time-variant column.
    
    _l12: value lagged 12 months
    _v12: 12-month moving average, lagged 12 months (avg of months 13-24 before)
    """
    df = df.copy()
    
    # Ensure sorted by admin_code and time
    df = df.sort_values([admin_col, year_col, month_col]).reset_index(drop=True)
    
    for col in time_variant_cols:
        # _l12: shift 12 months within each admin_code
        df[f'{col}_l12'] = df.groupby(admin_col)[col].shift(12)
        
        # _v12: 12-month rolling mean, then shift 12 months
        # rolling(12) on the original gives avg of current + past 11 months
        # shift(12) then pushes that to 12 months ago => avg of months 13-24 ago
        df[f'{col}_v12'] = (
            df.groupby(admin_col)[col]
              .transform(lambda x: x.rolling(12, min_periods=12).mean())
              .groupby(df[admin_col])
              .shift(12)
        )
    
    return df

# Usage:

df_forecast = create_lag_features(df_forecast, time_variant, admin_col='admin_code')


C:\Users\swl00\AppData\Local\Temp\ipykernel_4876\2964687930.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_v12'] = (
C:\Users\swl00\AppData\Local\Temp\ipykernel_4876\2964687930.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_l12'] = df.groupby(admin_col)[col].shift(12)
C:\Users\swl00\AppData\Local\Temp\ipykernel_4876\2964687930.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining 

In [14]:
# drop time_variant

df_forecast = df_forecast.drop(columns=time_variant)

In [15]:
df_forecast.columns.tolist()

['admin_code',
 'year',
 'month',
 'date',
 'AEZ_32000',
 'AEZ_34000',
 'AEZ_36000',
 'AEZ_38000',
 'AEZ_42000',
 'AEZ_7000',
 'crop',
 'distance_to_river',
 'elevation',
 'estimated_population',
 'lat',
 'lon',
 'market_access',
 'overall_phase',
 'phase1_percent',
 'phase2_percent',
 'phase3_percent',
 'phase4_percent',
 'phase5_percent',
 'popdensity',
 'range',
 'ruggedness',
 'sg_cec_5-15cm',
 'sg_cfvo_5-15cm',
 'sg_nitrogen_5-15cm',
 'sg_phh2o_5-15cm',
 'sg_soc_5-15cm',
 'slope',
 'AEZ_10000',
 'AEZ_12000',
 'AEZ_17000',
 'AEZ_19000',
 'AEZ_20000',
 'AEZ_25000',
 'AEZ_28000',
 'AEZ_30000',
 'AEZ_31000',
 'AEZ_33000',
 'AEZ_35000',
 'AEZ_4000',
 'AEZ_40000',
 'AEZ_43000',
 'AEZ_9000',
 'nightlight_mean_l12',
 'nightlight_mean_v12',
 'nightlight_std_l12',
 'nightlight_std_v12',
 'market_distance_l12',
 'market_distance_v12',
 'gini_l12',
 'gini_v12',
 'event_count_battles_l12',
 'event_count_battles_v12',
 'event_count_battles_w10_l12',
 'event_count_battles_w10_v12',
 'event_count

In [16]:
df_forecast = df_forecast.sort_values(['admin_code', 'year', 'month']).reset_index(drop=True)

# forward-fill overall_phase within each admin_code, then lag by 1 to avoid leakage
df_forecast['overall_phase'] = df_forecast['overall_phase'].apply(lambda x: 1 if x < 1 else (5 if x > 5 else x))
df_forecast['last_overall_phase'] = (
    df_forecast.groupby('admin_code')['overall_phase']
      .transform(lambda x: x.ffill().shift(1))
)

In [17]:
# drop area_id if all overall_phase are missing
df_forecast = df_forecast[~df_forecast.groupby('admin_code')['overall_phase'].transform(lambda x: x.isna().all())]

In [18]:
# drop if year==2021
df_forecast = df_forecast[df_forecast['year'] != 2021]

In [19]:
df_forecast = df_forecast[df_forecast['year'] != 2020]

In [22]:
#rename admin_code as area_id
df_forecast = df_forecast.rename(columns={'admin_code': 'area_id'})

In [46]:
del df
df = df_forecast.copy()
df_2025 = df[df['year'] == 2025]
df_2024 = df[df['year']!=2025]

# drop NAs in overall_phase in df_2024
df_2024 = df_2024.dropna(subset=['overall_phase'])
df = pd.concat([df_2024,df_2025])

# sort by area_id, date
df = df.sort_values(by=['area_id', 'date'])

In [47]:
df_origin = df.copy()
y_pred_test = pd.DataFrame()
model_stats = pd.DataFrame()
#select phase1_percent is not na
# Sort by region and date
df = df.sort_values(by=['area_id', 'date'])
#drop overall phase
df = df.drop(['overall_phase'], axis=1)
#for each region, set last observation to be test set
# create a series of new outcome, phase2_worse=phase2_percent+phase3_percent+phase4_percent+phase5_percent, phase3_worse=phase3_percent+phase4_percent+phase5_percent, phase4_worse=phase4_percent+phase5_percent, phase5_worse=phase5_percent
df['phase2_worse'] = df['phase2_percent'] + df['phase3_percent'] + df['phase4_percent'] + df['phase5_percent']
df['phase3_worse'] = df['phase3_percent'] + df['phase4_percent'] + df['phase5_percent']
df['phase4_worse'] = df['phase4_percent'] + df['phase5_percent']
df['phase5_worse'] = df['phase5_percent']
#drop phase2_percent, phase3_percent, phase4_percent, phase5_percent, phase1_percent
df = df.drop(['phase2_percent', 'phase3_percent', 'phase4_percent', 'phase5_percent', 'phase1_percent'], axis=1)
# Splitting the data
#test_df = df.groupby('area_id').tail(1)
#train_df = df.drop(test_df.index)
#test_df = test_df.drop(['area_id','date'], axis=1)
#train_df = train_df.drop(['area_id','date'], axis=1)

In [48]:
y_pred_test = pd.DataFrame()
# drop anything after 2022-01-01
#df = df[df['date'] < '2021-01-01']
shape_values_df_ensemble = pd.DataFrame()
df_result = pd.DataFrame()
#order unique_dates
y_pred_test=pd.DataFrame()
# for df,replace all inf and -inf as missing
df = df.replace([np.inf, -np.inf], np.nan)
for i in range(2, 6):
    train_df = df[(df['year'] != 2025)]
    test_df = df[df['year'] == 2025]
    train_df = train_df.drop(['date','area_id'], axis=1)
    identifier = test_df[['date','area_id']]
    test_df = test_df.drop(['date','area_id'], axis=1)
    train_df_new = train_df.drop(['phase{}_worse'.format(j) for j in range(2, 6) if j != i], axis=1)
    test_df_new = test_df.drop(['phase{}_worse'.format(j) for j in range(2, 6) if j != i], axis=1)
# drop rows with NaN in phase{}_percent
    train_df_new = train_df_new.dropna(subset=['phase{}_worse'.format(i)])
    test_index = test_df_new.index
    # Split into features and target
    X_train = train_df_new.drop('phase{}_worse'.format(i), axis=1)
    y_train = train_df_new['phase{}_worse'.format(i)]
    X_test = test_df_new.drop('phase{}_worse'.format(i), axis=1)
    y_test = test_df_new['phase{}_worse'.format(i)]
    #fews_ipc_ha_test = X_test['fews_ipc_ha']
    #X_train = X_train.drop(['fews_ipc_ha'], axis=1)
    #X_test = X_test.drop(['fews_ipc_ha'], axis=1)
    if i == 3:
        best_params_xgb_regressor = best_params_xgb_regressor_for_p3
    model = xgb.XGBRegressor(**best_params_xgb_regressor)
    model.fit(X_train, y_train)
    # Predictions
    y_pred = model.predict(X_test)
    # for y_pred_test, add a column to indicate the phase
    #y_pred_test = pd.concat([y_pred_test, pd.DataFrame({'y_pred': y_pred, 'y_test': y_test, 'phase': [i]*len(y_pred),'fews_ipc_ha':fews_ipc_ha_test,'test_index':test_index})], ignore_index=True)
    if i != 5:
        y_pred_test = pd.concat([y_pred_test, pd.DataFrame({'y_pred': y_pred, 'y_test': y_test, 'phase': [i]*len(y_pred)})], ignore_index=True)
    else:
        y_pred_test = pd.concat([y_pred_test, pd.DataFrame({'y_pred': y_pred, 'y_test': y_test, 'phase': [i]*len(y_pred)})], ignore_index=True)
        #concate identifier
        y_pred_test = pd.concat([y_pred_test, identifier], axis=1)
   


In [49]:
#save y_pred_test
y_pred_test.to_csv('y_pred_test.csv', index=False)

In [50]:
#drop missing in area_id
y_pred_test = y_pred_test.dropna(subset=['area_id'])

In [51]:
# drop y_test
y_pred_test = y_pred_test.drop(['y_test'], axis=1)

In [53]:
y_pred_test['indicator'] = y_pred_test['y_pred'] >= 0.2

In [54]:
y_pred_test['phase_ind'] = y_pred_test['indicator'] * y_pred_test['phase']

In [55]:
#group by date and area_id, extract the mas of phase_ind

y_pred_test_final = y_pred_test.groupby(['date', 'area_id'])['phase_ind'].max().reset_index()

In [56]:
df_identifier = df[['lat', 'lon','area_id']].drop_duplicates()

In [57]:
# merge y_pred_test_final with df_identifier
y_pred_test_final = y_pred_test_final.merge(df_identifier, on='area_id', how='left')

In [58]:
#drop phase_ind is missing
y_pred_test_final = y_pred_test_final.dropna(subset=['phase_ind'])

In [ ]:
y_pred_test_final.to_csv('y_pred_final_2025.csv', index=False)